# ML-07 — Baseline Action Score + Top-20 Review

**Lane 2 (refresh / opportunity scoring).** ML-05 built and leak-checked the feature vector
(108,254 eligible pages). This assignment builds the **transparent baseline** that every model
must beat — a hand-written rule an editor can read, with **reason codes** on every pick, a
**ranked queue**, and **precision@K** judged against the base rate.

Built with the `building-baselines` skill.

**The one-line claim:** a simple, readable rule ("has traffic + is old + position slipping")
flags 13k pages with **precision@20 = 0.85** against a **0.674 base rate** — a ~1.26x lift
from a rule with zero fitted weights. Its top-20 picks are genuine refresh candidates; the
misses are high-traffic pages that unexpectedly held steady.

## 1. My rule and its reason codes

**The rule, in plain words:** *a page is worth refreshing first if it used to get real traffic,
is getting old, and its average search position is slipping (weak).* All three must hold to be
flagged; the score then just orders flagged pages by how much traffic is at stake.

Transparent conditions (no fitted weights, thresholds read directly off the data):

| Condition | Test | Why |
|---|---|---|
| `has_traffic` | `imp_b ≥ 600` | a page with nothing to lose isn't a priority |
| `stale` | `age_days ≥ 180` | old content decays; ~half of pages are under 6 months |
| `position_slipping` | `pos_avg_b ≥ 12` | position 12+ is well off page-1 territory (1 = best, worse = higher number) |

**Score:** `score = has_traffic × stale × position_slipping × imp_b` — a readable product, so a
page is only scored > 0 when all three fire, and volume then decides order.

**Reason codes** (one on every pick, this is what makes the list trustworthy to a human):
`has_traffic`, `stale`, `position_slipping` — combined with `+`; pages with none get
`low_signals`.

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath("../scripts"))

import duckdb
import pandas as pd
import numpy as np
from datetime import timedelta

import hf_query

con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '" + hf_query.get_token() + "')")

REL = hf_query.REL
T = {
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "daily":   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

D_MAX = con.sql(f"SELECT MAX(report_date) FROM {T['daily']}").fetchone()[0]
t = D_MAX - timedelta(days=30)
b_lo = t - timedelta(days=30)

feat = con.sql(f"""
WITH win AS (
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_avg_position, gsc_data_available
    FROM {T['daily']} WHERE month IN ('{t:%Y-%m}', '{D_MAX:%Y-%m}')
),
agg AS (
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_b,
           SUM(CASE WHEN report_date >  DATE '{t}' THEN gsc_impressions ELSE 0 END) AS imp_f,
           SUM(CASE WHEN report_date <= DATE '{t}' THEN gsc_clicks ELSE 0 END) AS clk_b,
           SUM(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available THEN 1 ELSE 0 END) AS gsc_days_b,
           AVG(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available
                    AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0
                    THEN gsc_avg_position END) AS pos_avg_b,
           STDDEV(CASE WHEN report_date <= DATE '{t}' AND gsc_data_available
                    AND gsc_avg_position IS NOT NULL AND gsc_avg_position <> 0
                    THEN gsc_avg_position END) AS pos_vol_b
    FROM win GROUP BY 1, 2
),
j AS (
    SELECT a.client_hash_id, a.content_hash_id, a.imp_b, a.imp_f, a.clk_b, a.gsc_days_b,
           a.pos_avg_b, a.pos_vol_b,
           c.content_type, c.word_count, c.char_count, c.keyword_char_count,
           c.keyword_token_count, c.url_char_count, c.main_intent, c.competition_level,
           c.category_count, c.search_volume, c.backlinks,
           c.content_created_date, c.content_updated_date, c.is_deleted, c.is_published
    FROM agg a LEFT JOIN {T['content']} c USING (client_hash_id, content_hash_id)
)
SELECT *,
       DATE '{t}' - content_created_date AS age_days,
       DATE '{t}' - content_updated_date AS days_since_update,
       CASE WHEN imp_f < 0.8 * imp_b THEN 1 ELSE 0 END AS declined_30d
FROM j WHERE imp_b >= 100 AND gsc_days_b >= 15
""").df()

feat["ctr_b"] = feat.clk_b / feat.imp_b
print("eligible pages:", len(feat))
print("declined_30d base rate:", round(feat.declined_30d.mean(), 4))

# ---- the transparent rule ----
TRAF, OLD, SLIP = 600, 180, 12
df = feat.copy()
df["has_traffic"]       = (df.imp_b >= TRAF).astype(int)
df["is_old"]            = (df.age_days >= OLD).astype(int)
df["pos_weak"]          = (df.pos_avg_b >= SLIP).astype(int)

def reason(r):
    parts = []
    if r.has_traffic: parts.append("has_traffic")
    if r.is_old:      parts.append("stale")
    if r.pos_weak:    parts.append("position_slipping")
    return "+".join(parts) if parts else "low_signals"

df["reason"] = df.apply(reason, axis=1)
df["score"]  = df.has_traffic * df.is_old * df.pos_weak * df.imp_b

print("flagged pages (score > 0):", int((df.score > 0).sum()))
print("reason-code distribution (all eligible):")
print(df.reason.value_counts().to_string())

C:\Users\Bogdan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


eligible pages: 108254
declined_30d base rate: 0.6744


flagged pages (score > 0): 12930
reason-code distribution (all eligible):
reason
stale+position_slipping                22464
position_slipping                      18196
has_traffic                            15629
has_traffic+stale                      14474
has_traffic+stale+position_slipping    12930
has_traffic+position_slipping          10837
low_signals                             9342
stale                                   4382


## 2. Build the ranked queue

Rank everything by `score` (descending), write the queue to
`work/outputs/baseline_action_score.csv`, and evaluate **precision@K** against the base rate.

In [2]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df.declined_30d.values
print("base rate:", round(y.mean(), 4))
for k in [20, 50, 100, 200]:
    print(f"precision@{k}: {precision_at_k(df.score.values, y, k):.3f}")

queue = df[["client_hash_id", "content_hash_id", "imp_b", "age_days", "pos_avg_b",
            "score", "reason", "declined_30d"]].sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs(os.path.abspath("../outputs"), exist_ok=True)
queue.to_csv(os.path.abspath("../outputs/baseline_action_score.csv"), index=False)
print("wrote work/outputs/baseline_action_score.csv:", queue.shape)

base rate: 0.6744
precision@20: 0.850
precision@50: 0.780
precision@100: 0.810
precision@200: 0.775


wrote work/outputs/baseline_action_score.csv: (108254, 8)


## 3. Top-20 review

The top of the list is where bad rule-logic shows itself. For each: the action, the reason
code, a confidence note, and what would make it wrong.

In [3]:
top20 = queue.head(20)
print(top20[["score", "reason", "imp_b", "age_days", "pos_avg_b", "declined_30d"]].to_string())

       score                               reason     imp_b  age_days  pos_avg_b  declined_30d
0   151672.0  has_traffic+stale+position_slipping  151672.0       290  23.888427             1
1   121749.0  has_traffic+stale+position_slipping  121749.0       202  14.357788             0
2    98599.0  has_traffic+stale+position_slipping   98599.0       207  16.640345             1
3    97393.0  has_traffic+stale+position_slipping   97393.0       220  26.064530             1
4    96732.0  has_traffic+stale+position_slipping   96732.0       298  16.178146             1
5    86533.0  has_traffic+stale+position_slipping   86533.0       214  22.498653             1
6    84795.0  has_traffic+stale+position_slipping   84795.0       207  33.461259             1
7    78680.0  has_traffic+stale+position_slipping   78680.0       436  24.043852             0
8    74625.0  has_traffic+stale+position_slipping   74625.0       202  14.008192             1
9    72868.0  has_traffic+stale+position_slipping 

**Top-20 review:**

- **Action (all 20):** add to the editor's refresh queue — each is high-traffic, old, and
  sitting off page 1, so a refresh is the cheapest lever with the most traffic at stake.
- **Reason code:** every one is `has_traffic+stale+position_slipping` — the rule is consistent,
  which is what you want from a baseline.
- **Confidence / what would make it wrong:**
  - 17 of 20 actually declined in the next 30 days (precision 0.85 vs 0.674 base) — confident.
  - The 3 misses (rows 1, 7, 11) held steady despite fitting all three conditions. Their
    common trait is very high traffic (`imp_b` 60k–120k): heavily-trafficked old pages can
    stay flat even when off page 1. A rule keyed to raw volume does not see that nuance —
    that is exactly the kind of miss a learned model should beat, and a fair thing to hold
    against this baseline.

## 4. Weak picks + leakage check

**Weak picks found:** the three high-traffic misses above are the weakest logic. The rule's
`× imp_b` ordering over-trusts sheer volume, so among the flagged set the biggest pages get
ranked first even when they are the least likely to move. That is an expected, honest weakness —
not a reason to distrust precision@20.

**Leakage check (confirm no leaks):**

| Check | Status |
|---|---|
| Features strictly in `B` or static (`imp_b`, `age_days`, `pos_avg_b` all ≤ `t`) | clean |
| Label's own input (`imp_f`) used anywhere in the rule? | no — declined only in the label |
| Future-window / q90 aggregates used? | no |
| Product flags (`provider_used`, `model_used`) used? | no — excluded in ML-05 |
| Split / base rate reported with metrics? | yes — base rate 0.674 beside every precision@K |

The rule is frozen here; downstream model comparisons use this exact queue so nothing moves
to make the model look better.

## Self-check

- [x] Rule stated in plain words first (section 1), threshold values shown
- [x] Transparent score + reason codes on every pick
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv`
- [x] precision@K computed on the same data slice/labels the model will use; base rate beside it
- [x] Top-20 reviewed by hand; weak picks found and explained
- [x] Leakage check: no product flags, no future windows, no `imp_f`
- [x] No client names, URLs, or raw identifiers in any output
- [x] The notebook runs top to bottom with no errors
- [x] Committed to `work/notebooks/` — then submit repo URL on the ML-07 card